# Graph-of-Thoughts (GoT)

*Level 8 — Reasoning Strategies*

## Objective

Generalize Tree-of-Thought with the one operation a tree cannot represent: **aggregation** --
merging two promising-but-different branches into one synthesized node with both as parents,
instead of only ever branching and pruning. Also covers Hierarchical GoT (HGoT), the RAG-specific
descendant that decomposes a question into sub-questions and retrieves separate, real evidence
for each one.


In [1]:
import sys
from pathlib import Path

LEVEL_DIR = Path.cwd().parent
sys.path.insert(0, str(LEVEL_DIR))
sys.path.insert(0, str(LEVEL_DIR / "graph-of-thought"))

from reasoning_common.dataset import prepare
from reasoning_common.embed import OllamaEmbedder
from reasoning_common.llm import OllamaLLM
from reasoning_common.retrieval import DenseRetriever
from thought_graph import ThoughtGraph
from graph_search import graph_of_thought_search
from hgot_retrieval import hgot_answer, decompose_question

data = prepare()
embedder = OllamaEmbedder()
llm = OllamaLLM()
retriever = DenseRetriever.from_corpus(data.corpus, embedder=embedder)
print(f"corpus: {len(data.corpus)} facts, {len(data.questions)} questions")

corpus: 282 facts, 120 questions


## The full graph search, on the same Mount Fuji question

Same question `02_tree_of_thought.ipynb` used, so the two strategies are directly comparable on
identical retrieved evidence.

In [2]:
fuji_question = "Would the top of Mount Fuji stick out of the Sea of Japan?"
fuji_retrieved = retriever.search(fuji_question, top_k=5)
fuji_context = "\n".join(data.corpus[doc_id] for doc_id, _ in fuji_retrieved)

result = graph_of_thought_search(fuji_question, fuji_context, llm=llm)
print("Real answer: True")
print("Best reasoning path:")
for step in result["best_path"]:
    print(" -", step)
print("\nBest score:", result["best_score"])
print("Final answer:", result["answer"])
print("LLM calls:", result["llm_calls"])
print("Graph size (total nodes explored):", result["graph_size"])

Real answer: True
Best reasoning path:
 - The average depth of the Sea of Japan is greater than Mount Fuji's height
 - Since the average depth of the Sea of Japan is greater than Mount Fuji's height, and considering that the maximum depth of the Sea of Japan is still significantly higher than Mount Fuji's height, it can be inferred that regardless of whether we compare the maximum or average depth, Mount Fuji would not stick out of the surface of the Sea of Japan.

Best score: 0.8
Final answer: False
LLM calls: 7
Graph size (total nodes explored): 5


In [3]:
graph = result["graph"]
merge_nodes = [n for n in graph.graph.nodes if len(graph.parents(n)) >= 2]
print(f"Nodes with more than one parent (real merges): {len(merge_nodes)}")
for node_id in merge_nodes:
    print(f"  node {node_id}: {graph.text(node_id)!r}")
    print(f"    parents: {[graph.text(p) for p in graph.parents(node_id)]}")

Nodes with more than one parent (real merges): 1
  node 4: "Since the average depth of the Sea of Japan is greater than Mount Fuji's height, and considering that the maximum depth of the Sea of Japan is still significantly higher than Mount Fuji's height, it can be inferred that regardless of whether we compare the maximum or average depth, Mount Fuji would not stick out of the surface of the Sea of Japan."
    parents: ["The average depth of the Sea of Japan is greater than Mount Fuji's height", "Comparing the maximum depth of the Sea of Japan to Mount Fuji's height may provide insight into whether its top would stick out"]


## What I observed -- the same failure mode, now compounded by aggregation

Real answer: **True**. GoT's answer: **False** -- wrong again, at 7 LLM calls (7x Chain-of-
Thought's cost).

The root claim is, again, factually backwards: *"The average depth of the Sea of Japan is greater
than Mount Fuji's height"* -- the real numbers are 5,748ft (average depth) vs. 12,389ft (Mount
Fuji), so this is false, and the evaluator scored it 0.8 anyway, the same confident-and-wrong
pattern `02_tree_of_thought.ipynb` found in Tree-of-Thought.

**Aggregation made this worse, not better.** The merge step combined this wrong premise with a
second branch into one more elaborate, more confident-*sounding* paragraph ("regardless of
whether we compare the maximum or average depth...") that still reached the wrong conclusion --
a real, concrete instance of `graph_search.py`'s own design note: synthesis has no mechanism to
catch an error already present in what it is synthesizing, and can instead make it read as more
authoritative. The graph structure itself worked exactly as designed (one real two-parent merge
node, confirmed directly below) -- the mechanism is not the problem; the judgment quality feeding
it is.

## HGoT: decompose into sub-questions, retrieve separately for each

A genuinely different mechanism from plain GoT above: instead of reasoning over one fixed context
block, break the question into parts and retrieve **independently** for each part.

In [4]:
hgot_question = "Did the band Led Zeppelin own a prime number of gilded gramophones?"

sub_questions = decompose_question(hgot_question, n=3, llm=llm)
print("Sub-questions:")
for sq in sub_questions:
    print(" -", sq)

Sub-questions:
 - Is the number of gilded gramophones a prime number?
 - Was Led Zeppelin a real band?
 - Did they ever own any gramophones?


In [5]:
result = hgot_answer(hgot_question, retriever, data.corpus, n_subquestions=3, llm=llm)
print("Real answer: True")
print("Final answer:", result["answer"])
print("LLM calls:", result["llm_calls"])
print("\nCited evidence:")
for doc_id in result["cited_evidence_ids"]:
    print(" -", data.corpus.get(doc_id, "(not in corpus)"))
print("\nPer-subquestion answers:")
for r in result["sub_results"]:
    print(f"  Q: {r['sub_question']}")
    print(f"  A: {r['answer_text']}")
    print()

Real answer: True
Final answer: False
LLM calls: 5

Cited evidence:
 - 5 is a prime number A Grammy Award trophy is a gilded gramophone Led Zeppelin won 5 Grammy Awards
 - Seven is a prime number.
 - Megadeth has sold over 38 million records worldwide.
 - Dave Mustaine formed the band Megadeth in 1983 and is the lead vocalist.
 - Metallica's original lead guitarist was Dave Mustaine.
 - An inanimate object is one that is not alive in any way.
 - Salsa is a popular Latin American music genre that is heavily connected to dance.

Per-subquestion answers:
  Q: Is the number of gilded gramophones a prime number?
  A: The context does not provide any information about the number of gilded gramophones, nor does it imply that such a quantity is relevant to determining whether it's prime.

  Q: Was Led Zeppelin a band that existed before 2000?
  A: The context does not provide information about when Led Zeppelin was formed, but it mentions they won 5 Grammy Awards, which implies that the awards

## What I observed (HGoT)

Real answer: **True**. HGoT's answer: **False** -- wrong, matching this level's own earlier CLI
test on the exact same question (see the README) -- a real, reproducible weakness for this
specific question across two independent runs, not a one-off fluke.

Two more honest observations from this run specifically:

- **The sub-questions differ between the two decompose calls in this notebook** (cell above vs.
  the ones printed alongside each per-subquestion answer) -- `decompose_question` is not
  perfectly deterministic even at this level's low `temperature=0.2`, a real reminder that the
  same prompt can genuinely phrase its decomposition differently call to call.
- **Some of the cited evidence is plainly unrelated** (real facts about Megadeth, Dave Mustaine,
  and salsa music turned up as "cited evidence" for a question about Led Zeppelin and Grammy
  trophies) -- a vague sub-question like "did they ever own any gramophones" does not always
  retrieve well against this level's real 282-fact corpus, and HGoT's per-subquestion retrieval
  has no mechanism to notice when its own retrieved evidence is off-topic before reasoning over
  it anyway.

## Common Failure Modes -- confirmed, not just anticipated

See `02_tree_of_thought.ipynb` for the full Mount Fuji trace: the same state-evaluator
unreliability affects Graph-of-Thoughts too, since it reuses the identical evaluator. Aggregation
adds one more real risk of its own: the synthesis step is itself an LLM call, asked to combine two
reasoning branches into one -- if either branch was already subtly wrong, the synthesis has no
mechanism to catch that, and can instead write a single confident-sounding paragraph that inherits
the error while sounding more authoritative than either branch did alone.